In [ ]:
import gc
import torch
import cv2
from ultralytics import YOLO
import sqlite3
import os
import time
from datetime import datetime
from fastapi import FastAPI, File, UploadFile, HTTPException, Query
import numpy as np

In [ ]:
gc.collect()
torch.cuda.empty_cache()
print(torch.cuda.memory_allocated() / 1024**2, "MB")

0.0 MB


In [ ]:
save_folder = "saved_image"
os.makedirs(save_folder, exist_ok=True)

db_path = "detection_history.db"
conn  = sqlite3.connect(db_path)
cursor = conn.cursor()

In [ ]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS knife_alerts (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        timestamp TEXT,
        image_path TEXT,
        confidence REAL
    )
""")
conn.commit()

In [ ]:
app = FastAPI(
    title="Knife detection Api",
    description="API для обнаружения холодного оружия с использованием YOLOv8",
    version="1.0.0"
)

In [ ]:
best_model = YOLO(r"C:\Ml-itsc\MoneyApp\Yolo_knife\runs\detect\train-6\weights\best.pt")

In [ ]:
@app.post("/detect_image", summary="Обнаружение на изображение")
async def detect_image(file: UploadFile = File(...)):
    contents = await file.read()
    nparr = np.frombuffer(contents, np.uint8)
    img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

    results = best_model(img)
    detections = []

    knife_detect = False
    max_conf = 0.0

    for result in results:
        for box in result.boxes:
            conf = float(box.conf)
            class_id = int(box.cls[0])

            if conf > 0.6:
                knife_detect = True
                if conf > max_conf:
                    max_conf = conf
                
                bbox = box.xyxy[0].tolist()
                detections.append({
                    "class_id": class_id,
                    "confidence": round(conf, 4),
                    "bbox": [round(x, 1) for x in bbox]
                })

    saved_path = None        
    if knife_detect:
        annotade_img = results[0].plot()
        now = datetime.now()

        time_str = now.strftime("%Y%m%d_%H%M%S_%f") 
        image_name = f"knife_{time_str}.jpg"
        image_path = os.path.join(save_folder, image_name)

        cv2.imwrite(image_path, annotade_img)

        formatted_time = now.strftime("%Y-%m-%d %H:%M:%S")
        cursor.execute("""
            INSERT INTO knife_alerts (timestamp, image_path, confidence)
            VALUES (?, ?, ?)
        """, (formatted_time, image_path, max_conf))

        conn.commit()
        conn.close()
    
    return {
        "knife_detected": knife_detect,
        "max_confidence": round(max_conf, 4) if knife_detect else 0.0,
        "detections": detections,
        "saved_to_db": knife_detect,
        "saved_image_path": saved_path
    }

        

    

In [ ]:
@app.get("/events", summary="Получить историю событий")
async def get_events(
    limit: int = Query(10, description="Количество записей для вывода", ge=1, le=100),
    offset: int = Query(0, description="Смещение для пагинации", ge=0)
):
    """
    Возвращает список зафиксированных инцидентов из базы данных SQLite с пагинацией.
    """
    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row  # Возвращает строки в виде словарей
    cursor = conn.cursor()
    
    cursor.execute("""
        SELECT id, timestamp, image_path, confidence 
        FROM knife_alerts 
        ORDER BY id DESC 
        LIMIT ? OFFSET ?
    """, (limit, offset))
    
    rows = cursor.fetchall()
    conn.close()

    # Формируем красивый JSON-ответ
    events = [dict(row) for row in rows]
    return {
        "total_returned": len(events),
        "limit": limit,
        "offset": offset,
        "events": events
    }

In [ ]:
import asyncio
import uvicorn

# Настраиваем конфигурацию Uvicorn
config = uvicorn.Config(app, host="127.0.0.1", port=8000)
server = uvicorn.Server(config)

# Запускаем сервер напрямую в текущем цикле событий Jupyter через await
await server.serve()
#http://127.0.0.1:8000/docs

INFO:     Started server process [33300]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)
INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [33300]
